# M3.5 — A/B Experiment Tracking (Member 3)

**What this notebook does:** prepares a fair comparison, creates plots and CSV summaries **from actual A/B logs once provided**. It does not train models or invent results. The data-leakage review is not yet signed off.

**Run Steps 1–4 now.** Steps 5–8 need the completed A/B runs and approved split. This notebook is independent of TensorFlow and the 2.7 GB HDF5 for preliminary work.

### Step 1 — Locate M3.5 files and load the code
Upload `member3_m35_package.zip` as a private Kaggle dataset and attach it under **Add Input**. Kaggle may extract its contents automatically. The cell below accepts either form.

In [ ]:
import os, sys, zipfile, shutil
from pathlib import Path
WORK = Path('/kaggle/working/member3_m35_work')
WORK.mkdir(parents=True,exist_ok=True)
inputs = Path('/kaggle/input')
packages = list(inputs.rglob('member3_m35_package.zip'))
if packages:
    with zipfile.ZipFile(packages[0]) as z:
        z.extractall(WORK)
else:
    for fn in ['member3_m35.py','test_member3_m35.py']:
        copies=list(inputs.rglob(fn))
        if not copies:
            raise FileNotFoundError(f'{fn} missing. Attach the M3.5 Kaggle dataset first.')
        shutil.copy2(copies[0],WORK/fn)
sys.path.insert(0,str(WORK))
import member3_m35 as m35
print('M3.5 module:',m35.__file__)
print('Files:',[p.name for p in WORK.iterdir()])

### Step 2 — Run M3.5 tests
These tests use **artificial CSV fixtures** to verify the code and do not represent model performance.

In [ ]:
import subprocess
subprocess.run([sys.executable,'-m','unittest','test_member3_m35','-v'],cwd=WORK,check=True)
print('PASS: M3.5 software tests completed')

### Step 3 — Produce A/B configuration review templates
The draft files intentionally have `null` for unset settings and `false` for unchecked evidence. This blocks premature final-report claims.

In [ ]:
TEMPLATES = Path('/kaggle/working/member3_m35_review_templates')
DATASET_SHA = 'a75906f8e3eae0150e3c9de48c7dd74faae3a823130e0030d27d6866a8ffbed4' # reported by team's handoff; confirm actual file before approving
m35.prepare_template(TEMPLATES, dataset_sha256=DATASET_SHA)
for p in sorted(TEMPLATES.glob('*.json')):
    print(p.name)
print('Template comparison status is PENDING_RUNS, not measured results.')

### Step 4 — Review the settings that still require a team decision
Consult Member 2: split approval, blank image 11475, corrected NPZ integration, resolution/steps/sigma/pseudo-label policies and B `lambda_rec`. Preserve the exact `corrected_split_indices.npz` checksum in both run metadata files.

In [ ]:
from pprint import pprint
print('Required *matching* A/B keys:')
pprint(m35.REQUIRED_MATCH_KEYS)
print('Evidence switches that must be verified, not guessed:')
pprint(('dataset_hash_verified','approved_split_integrated','external_cutouts_excluded','fresh_independent_start','team_split_approved'))
print('B lambda_rec must be positive; A must have no reconstruction term.')

### Step 5 — Locate the actual completed training runs (only after Member 2 runs them)
The filenames `val.csv` and `loss.csv` are produced by the trainer. Their exact folders will depend on the run settings. If no paths print, the logs are not attached to this notebook yet.

In [ ]:
roots=[Path('/kaggle/working'),Path('/kaggle/input')]
found=[]
for base in roots:
    if base.exists():
        for p in base.rglob('*.csv'):
            if p.name in ('val.csv','loss.csv','diagnostics.csv'):
                found.append(p)
print('Found relevant CSV files:')
for p in found[:60]: print(p)
if not found: print('No completed A/B logs found in this session. Stop at Step 4 for now.')

### Step 6 — Configure paths to *real* A and B logs
**Leave `RUN_REAL_COMPARISON=False` until both full runs are available.** Do not use the 5-step smoke logs as final A/B research results. Set metadata JSON paths to reviewed run configs created from approved records and runtime evidence, **not template defaults**.

In [ ]:
RUN_REAL_COMPARISON = False
VAL_A = ''   # actual .../A/val.csv
VAL_B = ''   # actual .../B/val.csv
LOSS_A = ''  # actual .../A/loss.csv
LOSS_B = ''  # actual .../B/loss.csv
CONFIG_A = ''  # A reviewed and fully completed JSON
CONFIG_B = ''  # B reviewed and fully completed JSON
DIAGNOSTIC_ONLY = False  # set True ONLY to inspect unapproved pilot runs; output will be explicitly marked
OUT = Path('/kaggle/working/member3_m35_real_results')
print('Run comparison enabled:',RUN_REAL_COMPARISON)

### Step 7 — Generate plots/validation comparison ONLY from actual CSVs
A reportable comparison is blocked if settings or evidence are missing. Diagnostic plots are labelled non-reportable. No test set is accessed.

In [ ]:
if RUN_REAL_COMPARISON:
    paths=[VAL_A,VAL_B,LOSS_A,LOSS_B]
    assert all(paths) and all(Path(p).is_file() for p in paths), 'Enter real A/B CSV paths first'
    ca=m35.load_json(CONFIG_A) if CONFIG_A else None
    cb=m35.load_json(CONFIG_B) if CONFIG_B else None
    result=m35.generate_validation_report(VAL_A,VAL_B,LOSS_A,LOSS_B,OUT,
        config_a=ca,config_b=cb,allow_unapproved=DIAGNOSTIC_ONLY)
    print('Result STATUS:',result['status'])
    print('Fairness:',result['fairness'])
    print('Saved outputs:', [p.name for p in OUT.iterdir()])
else:
    print('SKIPPED. Matched real A/B runs and split approval are still pending.')

### Step 8 — Export a handoff ZIP
The output contains configuration-review templates and M3.5 code/test files. Real plots are included only if Step 7 was run against actual logs.

In [ ]:
import shutil
from IPython.display import display,FileLink
BUNDLE=Path('/kaggle/working/member3_m35_export')
BUNDLE.mkdir(parents=True,exist_ok=True)
for file in TEMPLATES.glob('*.json'): shutil.copy2(file,BUNDLE/file.name)
for name in ('member3_m35.py','test_member3_m35.py','member3_m35_handoff.md','README_M35.md'):
    source=WORK/name
    if source.exists(): shutil.copy2(source,BUNDLE/name)
if OUT.exists():
    dest=BUNDLE/'real_outputs';dest.mkdir(exist_ok=True)
    for p in OUT.iterdir():
        if p.is_file(): shutil.copy2(p,dest/p.name)
zip_path=shutil.make_archive('/kaggle/working/member3_m35_results','zip',BUNDLE)
print('Created:',zip_path)
display(FileLink(zip_path))

## M3.5 completion criteria

- ✅ Reporter and config-check code written, 12 synthetic software tests passed locally.
- ⬜ Team reviews and approves the M3.2 conservative split and constant-value image 11475.
- ⬜ Member 2 merges and demonstrates checksum-protected corrected NPZ loading.
- ⬜ A/B full settings decided and frozen; B `lambda_rec` specified.
- ⬜ Independent equivalent A/B runs completed; **real** `val.csv` and `loss.csv` provided.
- ⬜ Real comparison generated and reviewed; final held-out test only after checkpoint selection is frozen.

**No final model accuracy can be claimed before the last steps.**